In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Subset, Dataset, WeightedRandomSampler
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from util import filter_data, seed_everything
from util import mask_crop as mask_crop_fn
from validate import val_model, ValLoaderWrapper
from loader import QSM_c1_Dataset as QSM_RAM_Dataset
from networks import QSMDecoder, ResNetWrapper

# ============================================================
# EXPERIMENT CONFIG & DIFFUSION SETUP
# ============================================================
EXP_NAME = "diff_mixup_fair_sens_080" 
device = 'cuda:0'
LIMIT_SUBS = None 
CACHE_PATH = 'qsm_preprocessed_cache.pt'
LOAD_FROM_CACHE = True 
seed = 0
seed_everything(0)

TIMESTEPS = 100
betas = torch.linspace(0.0001, 0.01, TIMESTEPS).to(device)
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

def apply_diffusion_noise(x_0, t):
    noise = torch.randn_like(x_0)
    sqrt_alphas_cumprod_t = torch.sqrt(alphas_cumprod[t]).view(-1, 1, 1, 1)
    sqrt_one_minus_alphas_cumprod_t = torch.sqrt(1. - alphas_cumprod[t]).view(-1, 1, 1, 1)
    return sqrt_alphas_cumprod_t * x_0 + sqrt_one_minus_alphas_cumprod_t * noise

# ============================================================
# DATA PREPARATION
# ============================================================
nii_path = '/data2/ali/dbs/qsm/'
seg_path = '/data2/ali/dbs/seg_ps/'
file_dir = '/data2/ali/dbs/dbs_03292024.csv'
cv_features = {'Age', 'Sex', 'Ethnicity', 'Race', 'Disease Duration (year)', ' pre op levadopa equivalent dose (mg)', ' Test medication status', ' OFF (pre-dbs updrs)', ' ON (pre-dbs updrs)'}
all_needed_cols = cv_features | {'CORNELL ID', ' OFF meds ON stim 6mo'}
motor_df = filter_data(file_dir, all_needed_cols, True)
for col in [' OFF (pre-dbs updrs)', ' ON (pre-dbs updrs)', ' OFF meds ON stim 6mo']:
    motor_df[col] = pd.to_numeric(motor_df[col], errors='coerce')
motor_df = motor_df.dropna(subset=[' OFF (pre-dbs updrs)', ' OFF meds ON stim 6mo'])
improvement_ratios = (motor_df[' OFF (pre-dbs updrs)'] - motor_df[' OFF meds ON stim 6mo']) / motor_df[' OFF (pre-dbs updrs)']
label_map = {str(int(row['CORNELL ID'])): (1 if ratio >= 0.30 else 0) for (_, row), ratio in zip(motor_df.iterrows(), improvement_ratios)}
cols_to_norm = ['Age', 'Disease Duration (year)', ' OFF (pre-dbs updrs)', ' pre op levadopa equivalent dose (mg)']
for col in cols_to_norm:
    motor_df[col] = pd.to_numeric(motor_df[col], errors='coerce')
    col_mean, col_std = motor_df[col].mean(), motor_df[col].std()
    motor_df[col] = (motor_df[col] - col_mean) / (col_std + 1e-8)
clinical_dict = {str(int(row['CORNELL ID'])): row[list(cv_features)].values.astype(np.float32) for _, row in motor_df.iterrows()}
full_dataset = QSM_RAM_Dataset(nii_path, seg_path, mask_crop_fn, clinical_dict, label_map, limit=LIMIT_SUBS, cache_path=CACHE_PATH, load_cache=LOAD_FROM_CACHE, return_index=True)
actual_clin_dim = next(iter(clinical_dict.values())).shape[0]
full_dataset.clin_dim = actual_clin_dim
all_cached_ids = {str(k) for k in full_dataset.volumes.keys()}
label_keys = {str(k) for k in label_map.keys()}
unique_labeled_subs = np.array(sorted(list(all_cached_ids & label_keys)))
unique_sub_labels = np.array([label_map[sid] for sid in unique_labeled_subs])
unlabeled_ids = np.array(list(all_cached_ids - label_keys))
qsm_aug = transforms.Compose([transforms.RandomRotation(degrees=15), transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05))])

# ============================================================
# CROSS-VALIDATION LOOP
# ============================================================
all_split_best_metrics = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

for split, (t_p_idx, v_p_idx) in enumerate(skf.split(unique_labeled_subs, unique_sub_labels)):
    train_subs, val_subs = unique_labeled_subs[t_p_idx], unique_labeled_subs[v_p_idx]
    t_idx = [i for i, s in enumerate(full_dataset.samples) if str(s['sub_id']) in set(train_subs)]
    v_idx = [i for i, s in enumerate(full_dataset.samples) if str(s['sub_id']) in set(val_subs)]
    pt_subs = np.concatenate([unlabeled_ids, train_subs])
    pt_idx = [i for i, s in enumerate(full_dataset.samples) if str(s['sub_id']) in set(pt_subs)]

    train_labels = [label_map[str(full_dataset.samples[i]['sub_id'])] for i in t_idx]
    class_counts = np.bincount(train_labels)
    class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    sampler = WeightedRandomSampler([class_weights[l] for l in train_labels], 2*len(t_idx))

    pt_loader = DataLoader(Subset(full_dataset, pt_idx), batch_size=48, shuffle=True)
    t_loader = DataLoader(Subset(full_dataset, t_idx), batch_size=48, sampler=sampler)
    v_loader = DataLoader(Subset(full_dataset, v_idx), batch_size=48, shuffle=False)

    base_resnet = models.resnet18(weights='IMAGENET1K_V1')
    base_resnet.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    model = ResNetWrapper(base_resnet, clinical_dim=actual_clin_dim).to(device)
    decoder = QSMDecoder(feat_dim=512).to(device)

    # --- A. DIFFUSION PRETRAINING ---
    optimizer_pt = torch.optim.Adam(list(model.base_model.parameters()) + list(decoder.parameters()), lr=1e-4)
    criterion_pt = nn.MSELoss()
    full_dataset.train_mode, full_dataset.transform = True, qsm_aug
    for pt_epoch in range(10):
        model.base_model.train(); decoder.train()
        for imgs, clin, _, _ in pt_loader:
            imgs, clin = imgs.to(device), clin.to(device)
            t = torch.randint(0, TIMESTEPS, (imgs.shape[0],), device=device).long()
            noisy_imgs = apply_diffusion_noise(imgs, t)
            optimizer_pt.zero_grad()
            _, feats = model(noisy_imgs, clin)
            recon = decoder(feats)
            criterion_pt(recon, imgs).backward(); optimizer_pt.step()

    # --- B. FINE-TUNING (Targeting 0.8 Sensitivity - Manifold Mixup) ---
    for param in model.base_model.parameters(): param.requires_grad = False
    optimizer = torch.optim.Adam(model.fusion.parameters(), lr=5e-5, weight_decay=1e-3)
    
    START_T, END_T, MAX_EPOCHS = 50, 1, 40 
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-7)
    loss_fn = nn.CrossEntropyLoss().to(device) # Keeping standard loss for fair comparison
    
    best_f1, best_metrics_this_split, patience = 0, None, 0
    os.makedirs(f"weights/{EXP_NAME}", exist_ok=True)

    for epoch in range(MAX_EPOCHS):
        model.train(); full_dataset.train_mode = True
        
        # Super-Convex Decay: Plateaus for 15 epochs to force feature robust extraction
        if epoch < 15:
            current_max_t = START_T
        else:
            decay_factor = 1 - ((epoch - 15) / (MAX_EPOCHS - 15))**0.2
            current_max_t = int(max(END_T, START_T * decay_factor))
        
        for imgs, clin, lbls, _ in t_loader:
            imgs, clin, lbls = imgs.to(device), clin.to(device), lbls.to(device)
            
            # Stochastic Noise Persistence
            t_val = START_T if torch.rand(1).item() < 0.25 else current_max_t
                
            if t_val > 1:
                jitter = torch.randint(-3, 4, (1,)).item()
                t_limit = max(2, min(TIMESTEPS-1, t_val + jitter))
                t = torch.randint(1, t_limit, (imgs.shape[0],), device=device).long()
                noisy_imgs = apply_diffusion_noise(imgs, t)
            else:
                noisy_imgs = imgs
            
            optimizer.zero_grad()
            
            # IMPLEMENTATION: Manifold Mixup (Alpha=0.2 for soft blending)
            if torch.rand(1).item() < 0.5:
                lam = np.random.beta(0.4, 0.4)
                batch_size = imgs.size(0)
                index = torch.randperm(batch_size).to(device)
                
                mixed_imgs = lam * noisy_imgs + (1 - lam) * noisy_imgs[index]
                mixed_clin = lam * clin + (1 - lam) * clin[index]
                
                logits, _ = model(mixed_imgs, mixed_clin)
                loss = lam * loss_fn(logits, lbls) + (1 - lam) * loss_fn(logits, lbls[index])
            else:
                logits, _ = model(noisy_imgs, clin)
                loss = loss_fn(logits, lbls)
                
            loss.backward()
            optimizer.step()
        
        scheduler.step()
        model.eval(); full_dataset.train_mode, full_dataset.transform = False, None
        wrapped_v_loader = ValLoaderWrapper(v_loader)
        
        # VALIDATION: Standard 0.5 Threshold
        m = val_model(wrapped_v_loader, device, model, loss_fn, v_loader.dataset, threshold=0.5)
        
        current_f1 = 2*(m[2]*m[3])/(m[2]+m[3]) if (m[2]+m[3])>0 else 0
        if current_f1 > best_f1:
            best_f1, best_metrics_this_split, patience = current_f1, m, 0
            torch.save(model.state_dict(), f"weights/{EXP_NAME}/split_{split}.pth")
        else:
            patience += 1

        print(f"S{split} E{epoch} | MaxT: {current_max_t} | Sens: {m[3]:.4f} | Spec: {m[4]:.4f} | F1: {current_f1:.4f}")
        if patience >= 12: break 
    
    if best_metrics_this_split is not None:
        all_split_best_metrics.append(best_metrics_this_split)

# ============================================================
# FINAL SUMMARY
# ============================================================
final_metrics = np.array(all_split_best_metrics)
avg_metrics, std_metrics = np.mean(final_metrics, axis=0), np.std(final_metrics, axis=0)
print("\n" + "="*45 + "\nTARGET 0.80 SENSITIVITY (MIXUP FAIR)\n" + "="*45)
names = ["Loss", "Accuracy", "Precision", "Sensitivity", "Specificity", "AUC"]
for i, name in enumerate(names):
    print(f"{name:<15} : {avg_metrics[i]:.4f} ± {std_metrics[i]:.4f}")
print("="*45)

Keeping CORNELL ID
Keeping Age
Keeping Sex
Keeping Ethnicity
Keeping Race
Keeping Disease Duration (year)
Keeping  OFF (pre-dbs updrs)
Keeping  ON (pre-dbs updrs)
Keeping  pre op levadopa equivalent dose (mg)
Keeping  Test medication status
Keeping  OFF meds ON stim 6mo
Loaded cache with 7776 slices from 108 subjects
S0 E0 | MaxT: 50 | Sens: 0.2500 | Spec: 0.9389 | F1: 0.3564
S0 E1 | MaxT: 50 | Sens: 0.4444 | Spec: 0.7958 | F1: 0.4547
S0 E2 | MaxT: 50 | Sens: 0.3264 | Spec: 0.8514 | F1: 0.3845
S0 E3 | MaxT: 50 | Sens: 0.6667 | Spec: 0.7000 | F1: 0.5517
S0 E4 | MaxT: 50 | Sens: 0.6007 | Spec: 0.7000 | F1: 0.5111
S0 E5 | MaxT: 50 | Sens: 0.5938 | Spec: 0.7125 | F1: 0.5135
S0 E6 | MaxT: 50 | Sens: 0.6771 | Spec: 0.7056 | F1: 0.5612
S0 E7 | MaxT: 50 | Sens: 0.5764 | Spec: 0.7444 | F1: 0.5204
S0 E8 | MaxT: 50 | Sens: 0.5903 | Spec: 0.7486 | F1: 0.5321
S0 E9 | MaxT: 50 | Sens: 0.6979 | Spec: 0.7403 | F1: 0.5947
S0 E10 | MaxT: 50 | Sens: 0.7361 | Spec: 0.7264 | F1: 0.6083
S0 E11 | MaxT: 50 | 